In [15]:
import os

import chromadb
import dotenv
from agents import Agent, Runner, function_tool, trace, WebSearchTool
from agents.mcp import MCPServerStreamableHttp

dotenv.load_dotenv()

True

set up RAG Database

In [16]:
chroma_client = chromadb.PersistentClient(path="../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

Set up calorie look up tool

In [21]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int=3) -> str:
    """
    Tool function for a RAG  database to look up calorie information  for the  specific food items but not for meals
    
    Args:
       query: The food item to look up
       max_results: maximum number of results to return

    Returns:
       A string containing the nutrition information

    """
    results = nutrition_db.query(query_texts=[query],n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for query {query}"
    
    formatted_results = []

    for i,doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category= metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}) : {calories} calories per 100g "
        )

    return "Nutrition Information: \n" + "\n".join(formatted_results)
    


In [ ]:
#calorie_lookup_tool("banana",2)

'Nutrition Information: \nBanana (Fruits) : 89.0 calories per 100g \nBanana Juice ((Fruit)Juices) : 50.0 calories per 100g '

In [22]:
calorie_agent_with_search=Agent(
    name="Nutrition Assistant",
    instructions= """
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search the web to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if it's about a meal, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use web search to find the exact recipe and ingredients.
    * Once you know the ingredients, always use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 8 times.
     """,
    tools=[calorie_lookup_tool, WebSearchTool()]
)

In [25]:
with trace("Nutrition Assistant with Web Search tool"):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an american breakfast?"
    )
    print(result.final_output)

There isn’t a single value for “an American breakfast.” Calories vary widely by items. Rough ranges:

- Light: 300–500 kcal (e.g., egg + toast, coffee)
- Typical hearty: 500–800 kcal (eggs, bacon/sausage, hash browns or pancakes with syrup)
- Very large/restaurant portions: 900+ kcal

If you tell me the exact items (e.g., 2 eggs, bacon, toast, hash browns), I’ll estimate the calories for a single serving.
